In [1]:
import cv2
import numpy as np

# Load image in grayscale
image = cv2.imread(r"Brahmi2\test_images\sampleBrahmi.jpg", cv2.IMREAD_GRAYSCALE)

# Apply Gaussian blur to reduce noise
image_blurred = cv2.GaussianBlur(image, (5, 5), 0)


In [2]:
# Apply Otsu's thresholding to binarize the image
_, binary_image = cv2.threshold(image_blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# Display the binarized image (for debugging)
cv2.imshow("Binary Image", binary_image)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [3]:
# Find contours (text regions) in the image
contours, _ = cv2.findContours(binary_image, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Draw bounding boxes around detected text regions
for contour in contours:
    x, y, w, h = cv2.boundingRect(contour)
    cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)

# Show the segmented image (for debugging)
cv2.imshow("Segmented Image", image)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [4]:
# Resize image to match the model input size (e.g., 32x32 pixels)
image_resized = cv2.resize(binary_image, (32, 32))  # Adjust the size if necessary

# Show resized image (for debugging)
cv2.imshow("Resized Image", image_resized)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [5]:
import cv2
import numpy as np
from skimage.morphology import skeletonize
import torch

# Define the image path
image_path = r"Brahmi2\test_images\sampleBrahmi.jpg"

# Load the image
image = cv2.imread(image_path)
if image is None:
    print("Error: Could not load image.")
    exit()

# Convert to grayscale
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Apply median blur to reduce noise
blurred = cv2.medianBlur(gray, 3)

# Binarize using adaptive thresholding
thresh = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2)

# Skeletonize the image
inverted = 255 - thresh  # Invert for skeletonization (foreground should be white)
binary = inverted == 255  # Convert to boolean array
skeleton = skeletonize(binary)
skeleton_uint8 = (skeleton * 255).astype(np.uint8)

# Resize to model's input size (adjust as necessary based on your model's requirements)
input_size = 28  # Example size; replace with your model's actual input size
resized = cv2.resize(skeleton_uint8, (input_size, input_size), interpolation=cv2.INTER_AREA)

# Normalize to [0,1]
normalized = resized / 255.0

# Convert to PyTorch tensor
input_tensor = torch.from_numpy(normalized).float().unsqueeze(0).unsqueeze(0)  # (1, 1, 28, 28)

# Now, you can feed input_tensor to your model
# Example:
# model = YourModelClass()  # Replace with your actual model class
# model.load_state_dict(torch.load('epoch_10_checkpoint.pth'))
# model.eval()
# with torch.no_grad():
#     output = model(input_tensor)
#     # Process output to get recognized text

In [2]:
# test_brahmi.py
def brahmi_to_iast(text):
    if not text:
        return ""

    # Character mappings for Brahmi to IAST
    vowels = {
        '\U00011005': 'a', '\U00011006': 'ā', '\U00011007': 'i', '\U00011008': 'ī',
        '\U00011009': 'u', '\U0001100A': 'ū', '\U0001100B': 'ṛ', '\U0001100C': 'ṝ',
        '\U0001100D': 'ḷ', '\U0001100E': 'e', '\U0001100F': 'ai', '\U00011010': 'o',
        '\U00011011': 'au'
    }
    consonants = {
        '\U00011013': 'k', '\U00011014': 'kh', '\U00011015': 'g', '\U00011016': 'gh',
        '\U00011017': 'ṅ', '\U00011018': 'c', '\U00011019': 'ch', '\U0001101A': 'j',
        '\U0001101B': 'jh', '\U0001101C': 'ñ', '\U0001101D': 'ṭ', '\U0001101E': 'ṭh',
        '\U0001101F': 'ḍ', '\U00011020': 'ḍh', '\U00011021': 'ṇ', '\U00011022': 't',
        '\U00011023': 'th', '\U00011024': 'd', '\U00011025': 'dh', '\U00011026': 'n',
        '\U00011027': 'p', '\U00011028': 'ph', '\U00011029': 'b', '\U0001102A': 'bh',
        '\U0001102B': 'm', '\U0001102C': 'y', '\U0001102D': 'r', '\U0001102E': 'l',
        '\U0001102F': 'v', '\U00011030': 'ś', '\U00011031': 'ṣ', '\U00011032': 's',
        '\U00011033': 'h'
    }
    vowel_signs = {
        '\U00011038': 'ā', '\U00011039': 'i', '\U0001103A': 'i', '\U0001103B': 'ī',
        '\U0001103C': 'u', '\U0001103D': 'ū', '\U0001103E': 'ṛ', '\U0001103F': 'ṝ',
        '\U00011040': 'ḷ', '\U00011041': 'e', '\U00011042': 'e', '\U00011043': 'o',
        '\U00011044': 'au'
    }
    diacritics = {
        '\U00011000': 'ṃ', '\U00011001': 'ṃ', '\U00011002': 'ḥ', '\U00011046': ''
    }
    punctuation = {
        '\U0001104B': '॥', '\U0001104C': '।', '\U00011047': '.', '\U00011048': '..'
    }
    numerals = {
        '\U00011066': '0', '\U00011067': '1', '\U00011068': '2', '\U00011069': '3',
        '\U0001106A': '4', '\U0001106B': '5', '\U0001106C': '6', '\U0001106D': '7',
        '\U0001106E': '8', '\U0001106F': '9'
    }
    conjuncts = {
        '\U00011027\U00011046\U0001102D': 'pra',
        '\U00011022\U00011046\U0001102C': 'tya',
        '\U00011022\U00011046\U00011027': 'tp',
        '\U00011018\U00011046\U00011019': 'cch'
    }
    orthographic_corrections = {
        '\U00011044': ('au', 'o'),
        '\U0001103F\U00011046': ('ṝ', 'ṛ'),
        '\U0001100B\U00011046': ('ṛ', 'r')
    }

    all_chars = {**vowels, **consonants, **vowel_signs, **diacritics, **punctuation, **numerals}

    try:
        words = text.split()
        result = []

        for word in words:
            translated_word = ""
            i = 0
            has_final_virama = len(word) >= 2 and word[-1] == '\U00011046'
            word_to_process = word[:-1] if has_final_virama else word

            while i < len(word_to_process):
                if (i + 2 < len(word_to_process) and word_to_process[i:i+3] in conjuncts):
                    translated_word += conjuncts[word_to_process[i:i+3]]
                    i += 3
                    if (i < len(word_to_process) and word_to_process[i] in vowel_signs):
                        vowel = vowel_signs[word_to_process[i]]
                        if word_to_process[i] in orthographic_corrections:
                            incorrect, correct = orthographic_corrections[word_to_process[i]]
                            if vowel == incorrect:
                                vowel = correct
                        translated_word += vowel
                        i += 1
                    continue

                char = word_to_process[i]

                if char in numerals:
                    translated_word += numerals[char]
                    i += 1
                    continue

                if char in punctuation:
                    translated_word += punctuation[char]
                    i += 1
                    continue

                if char in vowels:
                    translated_word += vowels[char]
                    i += 1
                    continue

                if char in ['\U00011000', '\U00011001', '\U00011002']:
                    translated_word += diacritics[char]
                    i += 1
                    continue

                if char in consonants:
                    curr_cons = consonants[char]
                    i += 1
                    if (i < len(word_to_process) and word_to_process[i] == '\U00011046'):
                        i += 1
                        if (i < len(word_to_process) and word_to_process[i] in consonants):
                            next_cons = consonants[word_to_process[i]]
                            if curr_cons == 'c' and next_cons == 'c':
                                translated_word += 'cc'
                            else:
                                translated_word += curr_cons + next_cons
                            i += 1
                            if (i < len(word_to_process) and word_to_process[i] in vowel_signs):
                                vowel = vowel_signs[word_to_process[i]]
                                if word_to_process[i] in orthographic_corrections:
                                    incorrect, correct = orthographic_corrections[word_to_process[i]]
                                    if vowel == incorrect:
                                        vowel = correct
                                translated_word += vowel
                                i += 1
                            elif (i >= len(word_to_process) or word_to_process[i] != '\U00011046'):
                                translated_word += 'a'
                        else:
                            translated_word += curr_cons
                    else:
                        if (i < len(word_to_process) and word_to_process[i] in vowel_signs):
                            vowel = vowel_signs[word_to_process[i]]
                            if word_to_process[i] in orthographic_corrections:
                                incorrect, correct = orthographic_corrections[word_to_process[i]]
                                if vowel == incorrect:
                                    vowel = correct
                            translated_word += curr_cons + vowel
                            i += 1
                        else:
                            translated_word += curr_cons + 'a'
                    continue

                if char in vowel_signs:
                    vowel = vowel_signs[char]
                    if char in orthographic_corrections:
                        incorrect, correct = orthographic_corrections[char]
                        if vowel == incorrect:
                            vowel = correct
                    translated_word += vowel
                    i += 1
                    continue

                if char == '\U00011046':
                    i += 1
                    continue

                if '\U00011000' <= char <= '\U0001107F' and char not in all_chars:
                    raise ValueError(f"Unrecognized Brahmi character: {char} (Unicode: {hex(ord(char))})")

                translated_word += char
                i += 1

            if has_final_virama and translated_word.endswith('a'):
                translated_word = translated_word[:-1]

            result.append(translated_word)

        return ' '.join(result)

    except Exception as e:
        raise ValueError(f"Error transliterating Brahmi text: {str(e)}")

def iast_to_english(text):
    """Placeholder: Convert IAST to English translation."""
    try:
        translations = {
            "mama nāma": "my name",
            "pratītyasamutpādaṃ": "dependent origination",
            "akātreid": "at the edge",
            "piātāṇi": "protected"
        }
        return translations.get(text.lower(), f"No English translation found for '{text}'")
    except Exception as e:
        raise ValueError(f"Error translating IAST to English: {str(e)}")

def modi_to_iast(text):
    """Placeholder: Convert Modi script to IAST Latin transliteration."""
    if not text:
        return ""

    modi_mappings = {
        '\U00011628': 'ra', '\U0001162C': 'ma', '\U00011630': 'na', '\U00011632': 'sa',
        '\U00011624': 'ta', '\U0001162F': 'ṇa'
    }
    try:
        result = ""
        for char in text:
            if char in modi_mappings:
                result += modi_mappings[char]
            elif '\U00011600' <= char <= '\U0001167F':
                raise ValueError(f"Unrecognized Modi character: {char} (Unicode: {hex(ord(char))})")
            else:
                result += char
        return result
    except Exception as e:
        raise ValueError(f"Error transliterating Modi text: {str(e)}")

if __name__ == "__main__":
    test_text = "𑀅𑀦𑀼𑀢𑁆𑀧𑀸𑀤𑀫𑁆"
    result = brahmi_to_iast(test_text)
    print(f"Input: {test_text}\nOutput: {result}")

Input: 𑀅𑀦𑀼𑀢𑁆𑀧𑀸𑀤𑀫𑁆
Output: anutpādam
